# Representational Dimensionality Analysis

Quantifies how many effective dimensions the neural activity patterns span across stimulus conditions, per ROI, for the STgrid task.

Adapted from `FLT2/representational_dimensionality.ipynb`. STgrid is a single-session task (no
learning-stage runs), so the stage-split sections from the FLT2 version are dropped here. The
searchlight and cortical-surface-plot sections are also dropped for now (would need a
`dimensionality_searchlight.py` script and the `roi_surface_plotting.py` module ported from FLT2).

**Pipeline:**
1. Load masked GLMsingle beta CSVs (same files as `rsa_roi.py`)
2. Run de-mean -> average across repetitions -> one pattern per stimulus
3. PCA via covariance eigendecomposition
4. Compute participation ratio (PR) and cumulative explained variance
5. Save per-subject results (via `dimensionality_roi.py`); aggregate group-level here

## 1. Setup

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
import pingouin as pg

from glob import glob
from collections import defaultdict
from matplotlib.patches import Patch


In [ ]:
# ---- Global plot style ----
mpl.rcParams.update({
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 9,
    'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'legend.fontsize': 7, 'legend.title_fontsize': 8,
    'figure.dpi': 300, 'svg.fonttype': 'none',
})
sns.set_theme(style='ticks', rc=mpl.rcParams)

# Left/right hemisphere colors (acfMRI convention: blue = Left, red = Right)
HEMI_PALETTE = {'L': '#4477AA', 'R': '#CC6677'}

# ---- Paths ----
bidsroot      = '/bgfs/bchandrasekaran/krs228/data/FLT/data_denoised'
deriv_dir     = os.path.join(bidsroot, 'derivatives')
glmsingle_dir = os.path.join(deriv_dir, 'glmsingle')
out_dir       = os.path.join(glmsingle_dir, 'representational_dimensionality')
os.makedirs(out_dir, exist_ok=True)

space_label = 'MNI152NLin2009cAsym'

# ---- Subjects ----
# Same QA exclusions as task-stgrid_level-group.ipynb
ignore_subs = ['sub-FLT27',
               'sub-FLT07', # bad QA 11/14/23
               'sub-FLT11', 'sub-FLT12', 'sub-FLT20', 'sub-FLT30',
              ]

participants_df = pd.read_csv(os.path.join(bidsroot, 'participants.tsv'), sep='\t')
participants_df = participants_df[~participants_df.participant_id.isin(ignore_subs)].copy()
participants_df.sort_values(by=['participant_id'], ignore_index=True, inplace=True)
participants_df['subject'] = participants_df['participant_id'].str.replace('sub-', '', regex=False)

sub_list = list(participants_df['subject'])
print(f'{len(sub_list)} subjects:', sub_list)

# ---- ROIs (same order as rsa_roi.py) ----
# Subcortical ROIs (IC, MGN) excluded from group-level analysis/stats below: their PR/D_sheng
# estimates are confounded by very low tSNR and tiny voxel counts (~17-26 voxels), which can
# look like a genuine hierarchy effect but is more likely a data-quality artifact
# (see Section 8d-ii voxel-count confound check).
roi_list = [
    'L-HG', 'L-PT', 'L-PP', 'L-STGp', 'L-STGa',
    'L-ParsOp', 'L-ParsTri',
    'R-HG', 'R-PT', 'R-PP', 'R-STGp', 'R-STGa',
    'R-ParsOp', 'R-ParsTri',
]

region_pairs = [
    ('HG',      'L-HG',      'R-HG'),
    ('PP',      'L-PP',      'R-PP'),
    ('PT',      'L-PT',      'R-PT'),
    ('STGp',    'L-STGp',    'R-STGp'),
    ('STGa',    'L-STGa',    'R-STGa'),
    ('ParsOp',  'L-ParsOp',  'R-ParsOp'),
    ('ParsTri', 'L-ParsTri', 'R-ParsTri'),
]

# Anatomical ordering for plots: primary -> association
roi_order = [
    'L-HG', 'R-HG', 'L-PP', 'R-PP', 'L-PT', 'R-PT',
    'L-STGp', 'R-STGp', 'L-STGa', 'R-STGa',
    'L-ParsOp', 'R-ParsOp', 'L-ParsTri', 'R-ParsTri',
]

# ---- Regex (same as rsa_roi.py) ----
run_re  = re.compile(r'run-(\d+)')
stim_re = re.compile(r'(di\d+_[A-Za-z]+)')
rep_re  = re.compile(r'rep-(\d+)')


## 2. Helper Functions

In [ ]:
def load_roi_betas(sub_id, roi):
    """
    Load all trial beta vectors for one subject x ROI.

    Returns a nested dict:
        data[run_label][stim_label] = list of 1-D arrays (one per repetition)
    """
    roi_folder = os.path.join(
        glmsingle_dir, 'masked_statmaps',
        f'sub-{sub_id}', 'statmaps_masked', f'mask-{roi}'
    )
    csv_files = sorted(glob(os.path.join(roi_folder, '*.csv')))
    if not csv_files:
        return None

    data = defaultdict(lambda: defaultdict(list))

    for fpath in csv_files:
        fname   = os.path.basename(fpath)
        m_run   = run_re.search(fname)
        m_stim  = stim_re.search(fname)
        m_rep   = rep_re.search(fname)
        if m_run is None or m_stim is None or m_rep is None:
            continue

        run_label  = f'run-{m_run.group(1)}'
        stim_label = m_stim.group(1)

        try:
            vec = np.atleast_1d(np.genfromtxt(fpath))
        except Exception:
            continue
        if vec.ndim == 0 or np.all(np.isnan(vec)):
            continue

        data[run_label][stim_label].append(vec)

    return data if data else None


def build_stimulus_matrix(data):
    """
    Given nested dict from load_roi_betas():
      1. Run de-mean: subtract run mean pattern from every trial vector
      2. Average repetitions per stimulus
      3. Return (stimulus_labels, X) where X is (n_stimuli x n_voxels)
    """
    demeaned = defaultdict(list)

    for run_label, stim_dict in data.items():
        all_vecs = [v for vecs in stim_dict.values() for v in vecs]
        if not all_vecs:
            continue
        run_mean = np.mean(np.vstack(all_vecs), axis=0)

        for stim_label, vecs in stim_dict.items():
            for v in vecs:
                demeaned[stim_label].append(v - run_mean)

    if not demeaned:
        return None, None

    stimulus_labels = sorted(demeaned.keys())
    rows = []
    valid_labels = []
    for stim in stimulus_labels:
        vecs = demeaned[stim]
        if len(vecs) == 0:
            continue
        min_len = min(len(v) for v in vecs)
        mat = np.vstack([v[:min_len] for v in vecs])
        rows.append(mat.mean(axis=0))
        valid_labels.append(stim)

    if len(rows) < 2:
        return None, None

    return valid_labels, np.vstack(rows)


def compute_dimensionality(X, standardize_voxels=True):
    """
    Given stimulus-averaged matrix X (n_stimuli x n_voxels):
      1. Stimulus de-mean
      2. Optional voxel standardization (correlation-matrix PCA)
      3. Eigendecompose the gram matrix
      4. Compute PR and D_sheng (Sheng et al. 2022)

    Returns dict with keys: 'PR', 'D_sheng', 'eigenvalues', 'cumvar',
                             'n_stimuli', 'n_voxels'
    """
    n_stimuli, n_voxels = X.shape

    X = X - X.mean(axis=0, keepdims=True)

    if standardize_voxels:
        col_std = X.std(axis=0, ddof=1)
        col_std[col_std < 1e-10] = 1.0
        X = X / col_std

    if n_stimuli <= n_voxels:
        G = X @ X.T / (n_stimuli - 1)
    else:
        G = X.T @ X / (n_stimuli - 1)

    eigenvalues = np.linalg.eigvalsh(G)
    eigenvalues = eigenvalues[eigenvalues > 1e-10]
    eigenvalues = np.sort(eigenvalues)[::-1]

    PR = eigenvalues.sum()**2 / (eigenvalues**2).sum()
    cumvar = np.cumsum(eigenvalues) / eigenvalues.sum()

    assert 1.0 <= PR <= min(n_stimuli, n_voxels) + 1e-6, \
        f'PR={PR:.3f} out of expected range [1, {min(n_stimuli, n_voxels)}]'
    assert abs(cumvar[-1] - 1.0) < 1e-6, \
        f'cumvar does not reach 1.0 (got {cumvar[-1]:.6f})'

    # Sheng et al. (2022): N_k / prop_var_k  where k = eigenvalues >= 1
    kaiser_mask = eigenvalues >= 1
    N_k = int(kaiser_mask.sum())
    if N_k > 0:
        prop_var_k = eigenvalues[kaiser_mask].sum() / eigenvalues.sum()
        D_sheng = N_k / prop_var_k
    else:
        D_sheng = np.nan

    return {
        'PR': PR,
        'D_sheng': D_sheng,
        'eigenvalues': eigenvalues,
        'cumvar': cumvar,
        'n_stimuli': n_stimuli,
        'n_voxels': n_voxels,
    }


In [ ]:
def load_group_metric_df(directory, suffix):
    """
    Concatenate all per-subject `*_dimensionality_{suffix}.csv` files in `directory`
    into a single (subject x ROI) dataframe.
    """
    rows = []
    for fpath in sorted(glob(os.path.join(directory, f'*_dimensionality_{suffix}.csv'))):
        sub_id = re.search(r'sub-(FLT\d+)', os.path.basename(fpath)).group(1)
        df = pd.read_csv(fpath)
        df.insert(0, 'subject', sub_id)
        rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def _outline_only(ax):
    """Strip fill from box/strip plot artists, using the facecolor as the outline instead."""
    for collection in ax.collections:
        facecolor = collection.get_facecolor()
        collection.set_edgecolor(facecolor)
        collection.set_facecolor('none')
    for patch in ax.patches:
        facecolor = patch.get_facecolor()
        patch.set_edgecolor(facecolor)
        patch.set_facecolor('none')


def to_region_hemi_long(df, region_pairs, value_name):
    """Reshape a wide (subject x ROI) dataframe into long format with 'region'/'hemisphere'
    columns, using region_pairs = [(region, l_roi, r_roi), ...]."""
    rows = []
    for region, l_roi, r_roi in region_pairs:
        if l_roi in df.columns:
            rows.append(df[['subject', l_roi]].rename(columns={l_roi: value_name})
                        .assign(region=region, hemisphere='L'))
        if r_roi in df.columns:
            rows.append(df[['subject', r_roi]].rename(columns={r_roi: value_name})
                        .assign(region=region, hemisphere='R'))
    return pd.concat(rows, ignore_index=True)


## 3. Per-Subject, Per-ROI Computation

Split out to `dimensionality_roi.py` (run per-subject via `run_dimensionality_roi.sh` /
`loop_dimensionality_roi.sh`, same pattern as `rsa_roi.py`). Kept here inline (commented out)
for reference/debugging one subject at a time.

In [ ]:
'''
all_results = {}   # all_results[sub_id][roi] = compute_dimensionality() output

for sub_id in sub_list:
    print(f'\n--- {sub_id} ---')
    sub_results = {}

    for roi in roi_list:
        data = load_roi_betas(sub_id, roi)
        if data is None:
            print(f'  {roi}: no data')
            continue

        stim_labels, X = build_stimulus_matrix(data)
        if X is None:
            print(f'  {roi}: insufficient trials')
            continue

        try:
            result = compute_dimensionality(X)
        except AssertionError as e:
            print(f'  {roi}: sanity check failed - {e}')
            continue

        sub_results[roi] = result
        print(f'  {roi}: PR={result["PR"]:.2f}  '
              f'({result["n_stimuli"]} stimuli, {result["n_voxels"]} voxels)')

    all_results[sub_id] = sub_results

print('\nDone.')
'''


## 4. Save Results

In [ ]:
'''
for sub_id, sub_results in all_results.items():
    if not sub_results:
        continue

    # --- PR table ---
    pr_row = {roi: res['PR'] for roi, res in sub_results.items()}
    pd.DataFrame([pr_row]).to_csv(
        os.path.join(out_dir, f'sub-{sub_id}_dimensionality_PR.csv'),
        index=False
    )

    # --- Sheng et al. D table ---
    dsheng_row = {roi: res['D_sheng'] for roi, res in sub_results.items()}
    pd.DataFrame([dsheng_row]).to_csv(
        os.path.join(out_dir, f'sub-{sub_id}_dimensionality_Dsheng.csv'),
        index=False
    )

    # --- full eigenspectra ---
    npz_data = {}
    for roi, res in sub_results.items():
        npz_data[f'{roi}_eigenvalues'] = res['eigenvalues']
        npz_data[f'{roi}_cumvar']      = res['cumvar']
    np.savez(
        os.path.join(out_dir, f'sub-{sub_id}_dimensionality_eigenspectra.npz'),
        **npz_data
    )

print('Saved results to', out_dir)
'''


## 5. Single-Subject Visualization

Inspect one subject before aggregating.

In [ ]:
plot_sub = sub_list[0]   # change to inspect a different subject

_pr_csv     = os.path.join(out_dir, f'sub-{plot_sub}_dimensionality_PR.csv')
_dsheng_csv = os.path.join(out_dir, f'sub-{plot_sub}_dimensionality_Dsheng.csv')
_npz_path   = os.path.join(out_dir, f'sub-{plot_sub}_dimensionality_eigenspectra.npz')

_pr_row     = pd.read_csv(_pr_csv).iloc[0].to_dict()
_dsheng_row = pd.read_csv(_dsheng_csv).iloc[0].to_dict()
_npz_data   = np.load(_npz_path)

sub_res = {
    roi: {
        'PR':          _pr_row[roi],
        'D_sheng':     _dsheng_row.get(roi, np.nan),
        'eigenvalues': _npz_data[f'{roi}_eigenvalues'],
        'cumvar':      _npz_data[f'{roi}_cumvar'],
    }
    for roi in roi_list
    if roi in _pr_row and f'{roi}_eigenvalues' in _npz_data
}
rois_present = [r for r in roi_order if r in sub_res]

palette = sns.color_palette('tab20', len(rois_present))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Eigenspectrum --
ax = axes[0]
for color, roi in zip(palette, rois_present):
    eigs = sub_res[roi]['eigenvalues']
    ax.plot(np.arange(1, len(eigs) + 1), eigs, label=roi, color=color, alpha=0.8)
ax.set_xlabel('PC')
ax.set_ylabel('Eigenvalue')
ax.set_title(f'{plot_sub}: Eigenspectrum')
ax.legend(fontsize=7, ncol=2, loc='upper right')
sns.despine(ax=ax)

# -- Cumulative explained variance --
ax = axes[1]
for color, roi in zip(palette, rois_present):
    cumvar = sub_res[roi]['cumvar']
    ax.plot(np.arange(1, len(cumvar) + 1), cumvar, label=roi, color=color, alpha=0.8)
ax.axhline(0.9, color='k', linestyle='--', linewidth=0.8, alpha=0.5, label='90%')
ax.set_xlabel('PC')
ax.set_ylabel('Cumulative variance explained')
ax.set_title(f'{plot_sub}: Cumulative variance')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=7, ncol=2, loc='lower right')
sns.despine(ax=ax)

fig.tight_layout()
plt.show()


In [ ]:
# -- Single-subject PR bar chart: paired L/R grouped bars --

bar_width   = 0.35
group_gap   = 0.3

x_L, x_R, x_ticks, x_labels = [], [], [], []
x = 0
for region, l_roi, r_roi in region_pairs:
    x_L.append(x)
    x_R.append(x + bar_width)
    x_ticks.append(x + bar_width / 2)
    x_labels.append(region)
    x += bar_width * 2 + group_gap

fig, ax = plt.subplots(figsize=(12, 4))
for i, (region, l_roi, r_roi) in enumerate(region_pairs):
    ax.bar(x_L[i], sub_res[l_roi]['PR'] if l_roi in sub_res else np.nan,
           width=bar_width, color=HEMI_PALETTE['L'], edgecolor='white')
    ax.bar(x_R[i], sub_res[r_roi]['PR'] if r_roi in sub_res else np.nan,
           width=bar_width, color=HEMI_PALETTE['R'], edgecolor='white')

ax.legend(handles=[Patch(facecolor=HEMI_PALETTE['L'], label='Left'),
                   Patch(facecolor=HEMI_PALETTE['R'], label='Right')],
          loc='upper left', frameon=False)
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Participation ratio (PR)')
ax.set_title(f'{plot_sub}: Representational dimensionality per ROI')
sns.despine(ax=ax)
fig.tight_layout()
plt.show()


In [ ]:
# -- Single-subject Sheng dimensionality bar chart --
fig, ax = plt.subplots(figsize=(12, 4))
for i, (region, l_roi, r_roi) in enumerate(region_pairs):
    ax.bar(x_L[i], sub_res[l_roi]['D_sheng'] if l_roi in sub_res else np.nan,
           width=bar_width, color=HEMI_PALETTE['L'], edgecolor='white')
    ax.bar(x_R[i], sub_res[r_roi]['D_sheng'] if r_roi in sub_res else np.nan,
           width=bar_width, color=HEMI_PALETTE['R'], edgecolor='white')

ax.legend(handles=[Patch(facecolor=HEMI_PALETTE['L'], label='Left'),
                   Patch(facecolor=HEMI_PALETTE['R'], label='Right')],
          loc='upper left', frameon=False)
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Sheng dimensionality (D)')
ax.set_title(f'{plot_sub}: Sheng et al. dimensionality per ROI')
sns.despine(ax=ax)
fig.tight_layout()
plt.show()


## 6. Group-Level Aggregation

In [ ]:
# Load all per-subject PR CSVs
df_pr = load_group_metric_df(out_dir, 'PR')
print(f'Subjects loaded: {len(df_pr)}')
df_pr.head()


In [ ]:
# Group PR box+strip plot: paired L/R, outline-only boxes+points
region_order = [region for region, _, _ in region_pairs]
df_pr_long = to_region_hemi_long(df_pr, region_pairs, 'PR')

fig, ax = plt.subplots(figsize=(13, 5), dpi=200)
sns.stripplot(data=df_pr_long, x='region', y='PR', hue='hemisphere', order=region_order,
              hue_order=['L', 'R'], palette=HEMI_PALETTE, dodge=True, linewidth=0.5, size=3,
              legend=None, ax=ax, zorder=2)
sns.boxplot(data=df_pr_long, x='region', y='PR', hue='hemisphere', order=region_order,
            hue_order=['L', 'R'], palette=HEMI_PALETTE, dodge=True, linewidth=1, fliersize=0,
            ax=ax, zorder=1)
_outline_only(ax)

ax.legend(handles=[Patch(facecolor='none', edgecolor=HEMI_PALETTE['L'], label='Left'),
                   Patch(facecolor='none', edgecolor=HEMI_PALETTE['R'], label='Right')],
          loc='upper left', frameon=False)
ax.set_xticklabels(region_order, rotation=45, ha='right', fontsize=9)
ax.set_xlabel('')
ax.set_ylabel('Participation ratio (PR)')
ax.set_title(f'Group-level representational dimensionality (N={len(df_pr)})')
sns.despine(ax=ax)
fig.tight_layout()
plt.savefig(os.path.join(out_dir, 'group_PR_barplot.png'), dpi=300)
plt.show()


In [ ]:
# Load all per-subject D_sheng CSVs (computed directly per-ROI by dimensionality_roi.py --
# no searchlight step in this notebook, see header note)
df_ds = load_group_metric_df(out_dir, 'Dsheng')
print(f'Subjects loaded: {len(df_ds)}')
df_ds[roi_list].describe().round(2)


In [ ]:
# Group Sheng dimensionality box+strip plot: paired L/R, outline-only boxes+points
df_ds_long = to_region_hemi_long(df_ds, region_pairs, 'D_sheng')

fig, ax = plt.subplots(figsize=(13, 5), dpi=200)
sns.stripplot(data=df_ds_long, x='region', y='D_sheng', hue='hemisphere', order=region_order,
              hue_order=['L', 'R'], palette=HEMI_PALETTE, dodge=True, linewidth=0.5, size=3,
              legend=None, ax=ax, zorder=2)
sns.boxplot(data=df_ds_long, x='region', y='D_sheng', hue='hemisphere', order=region_order,
            hue_order=['L', 'R'], palette=HEMI_PALETTE, dodge=True, linewidth=1, fliersize=0,
            ax=ax, zorder=1)
_outline_only(ax)

ax.legend(handles=[Patch(facecolor='none', edgecolor=HEMI_PALETTE['L'], label='Left'),
                   Patch(facecolor='none', edgecolor=HEMI_PALETTE['R'], label='Right')],
          loc='upper left', frameon=False)
ax.set_xticklabels(region_order, rotation=45, ha='right', fontsize=9)
ax.set_xlabel('')
ax.set_ylabel('Sheng dimensionality (D)')
ax.set_title(f'Group-level Sheng dimensionality (N={len(df_ds)})')
sns.despine(ax=ax)
fig.tight_layout()
plt.savefig(os.path.join(out_dir, 'group_Dsheng_barplot.png'), dpi=200)
plt.show()


In [ ]:
# Summary table
rois_avail = [roi for roi in roi_order if roi in df_pr.columns]
group_mean = df_pr[rois_avail].mean()
group_sem  = df_pr[rois_avail].sem()

summary = pd.DataFrame({
    'ROI': rois_avail,
    'mean_PR': group_mean[rois_avail].values,
    'sem_PR':  group_sem[rois_avail].values,
    'n_subs':  df_pr[rois_avail].notna().sum().values,
})
summary.to_csv(os.path.join(out_dir, 'group_PR_summary.csv'), index=False)
print(summary.to_string(index=False))


## 7. Group Eigenspectrum Comparison

Average cumulative variance curves across subjects for each ROI.

In [ ]:
# Load per-subject cumvar arrays from saved NPZ files
cumvar_by_roi = defaultdict(list)
npz_files = sorted(glob(os.path.join(out_dir, '*_dimensionality_eigenspectra.npz')))
for fpath in npz_files:
    npz = np.load(fpath)
    for roi in roi_list:
        key = f'{roi}_cumvar'
        if key in npz:
            cumvar_by_roi[roi].append(npz[key])

# Interpolate to a common PC grid for averaging
n_pc_grid = 20
pc_grid   = np.arange(1, n_pc_grid + 1)

fig, ax = plt.subplots(figsize=(10, 5))
palette  = sns.color_palette('tab20', len(rois_avail))

for color, roi in zip(palette, rois_avail):
    curves = cumvar_by_roi.get(roi, [])
    if not curves:
        continue
    interp_curves = []
    for cv in curves:
        pcs = np.arange(1, len(cv) + 1)
        interp_curves.append(np.interp(pc_grid, pcs, cv, right=1.0))
    mean_cv = np.mean(interp_curves, axis=0)
    sem_cv  = np.std(interp_curves, axis=0) / np.sqrt(len(interp_curves))
    ax.plot(pc_grid, mean_cv, label=roi, color=color)
    ax.fill_between(pc_grid, mean_cv - sem_cv, mean_cv + sem_cv,
                    alpha=0.15, color=color)

ax.axhline(0.9, color='k', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_xlabel('PC')
ax.set_ylabel('Cumulative variance explained')
ax.set_title('Group-average cumulative explained variance per ROI')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=7, ncol=2, bbox_to_anchor=(1.01, 1), loc='upper left')
sns.despine(ax=ax)
fig.tight_layout()
plt.savefig(os.path.join(out_dir, 'group_cumvar_plot.png'), dpi=300)
plt.show()


## 8. Statistics: Comparing PR Across ROIs

Four complementary tests:
- **8a** Omnibus repeated-measures ANOVA across all ROIs
- **8b** All pairwise post-hoc comparisons (FDR-BH), visualised as a heatmap
- **8c** L vs R paired t-tests (FDR-BH across 7 pairs)
- **8d** Hierarchical level contrast: primary vs association (+ voxel-count confound check)

In [ ]:
# Reshape to long format; df_pr must already be loaded (Section 6)
df_long = (df_pr
    .melt(id_vars='subject', var_name='ROI', value_name='PR')
    .query('ROI in @roi_list')
    .dropna())

# 8a -- Omnibus repeated-measures ANOVA across all ROIs (Greenhouse-Geisser correction)
aov = pg.rm_anova(data=df_long, dv='PR', within='ROI',
                  subject='subject', correction=True)
display(aov[['Source', 'ddof1', 'ddof2', 'F', 'p-unc', 'p-GG-corr', 'ng2']])


In [ ]:
# 8b -- All pairwise post-hoc comparisons (FDR-BH)
posthoc = pg.pairwise_tests(data=df_long, dv='PR', within='ROI',
                             subject='subject', padjust='fdr_bh')
posthoc.to_csv(os.path.join(out_dir, 'stats_pairwise_posthoc.csv'), index=False)

# Build symmetric p-value matrix in roi_order for heatmap
rois_stat = [r for r in roi_order if r in df_long['ROI'].unique()]
n = len(rois_stat)
pmat = pd.DataFrame(np.ones((n, n)), index=rois_stat, columns=rois_stat)
for _, row in posthoc.iterrows():
    a, b, p = row['A'], row['B'], row['p-corr']
    if a in pmat.index and b in pmat.columns:
        pmat.loc[a, b] = p
        pmat.loc[b, a] = p
np.fill_diagonal(pmat.values, np.nan)

logp = -np.log10(pmat.astype(float))
sig_mask = pmat.astype(float) >= 0.05   # grey-out non-significant cells

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(logp, mask=sig_mask, cmap='Reds', vmin=0, vmax=4,
            linewidths=0.4, linecolor='white',
            cbar_kws={'label': '–log₁₀(p_FDR)'},
            ax=ax, square=True)
sns.heatmap(logp, mask=~sig_mask, cmap=['#e0e0e0'], vmin=0, vmax=1,
            linewidths=0.4, linecolor='white', cbar=False, ax=ax, square=True)
ax.set_title('Pairwise PR comparisons (–log₁₀ FDR-corrected p)', fontsize=11)
ax.tick_params(labelsize=8)
fig.tight_layout()
plt.savefig(os.path.join(out_dir, 'stats_pairwise_heatmap.png'), dpi=300)
plt.show()


In [ ]:
# 8c -- L vs R paired t-tests, FDR-BH corrected across region pairs
lv_results = []
for region, l_roi, r_roi in region_pairs:
    if l_roi not in df_pr.columns or r_roi not in df_pr.columns:
        continue
    vals = df_pr[[l_roi, r_roi]].dropna()
    res = pg.ttest(vals[l_roi], vals[r_roi], paired=True)
    res.insert(0, 'region', region)
    res['mean_diff'] = (vals[l_roi] - vals[r_roi]).mean()
    lv_results.append(res)

df_lv = pd.concat(lv_results, ignore_index=True)
_, df_lv['p-corr'] = pg.multicomp(df_lv['p-val'].values, method='fdr_bh')
df_lv.to_csv(os.path.join(out_dir, 'stats_LvR.csv'), index=False)
display(df_lv[['region', 'mean_diff', 'T', 'dof', 'p-val', 'p-corr', 'cohen-d']])


In [ ]:
# 8d -- Hierarchical level contrast: primary vs association
level_map = {
    'Primary':     ['L-HG', 'R-HG', 'L-PP', 'R-PP', 'L-PT', 'R-PT'],
    'Association': ['L-STGp', 'R-STGp', 'L-STGa', 'R-STGa',
                    'L-ParsOp', 'R-ParsOp', 'L-ParsTri', 'R-ParsTri'],
}
level_order = ['Primary', 'Association']

df_level = (pd.DataFrame(
    {level: df_pr[[r for r in rois if r in df_pr.columns]].mean(axis=1)
     for level, rois in level_map.items()}
).assign(subject=df_pr['subject'].values)
 .melt(id_vars='subject', var_name='level', value_name='PR')
 .dropna())

aov_level = pg.rm_anova(data=df_level, dv='PR', within='level',
                         subject='subject', correction=True)
ph_level  = pg.pairwise_tests(data=df_level, dv='PR', within='level',
                               subject='subject', padjust='fdr_bh')
ph_level.to_csv(os.path.join(out_dir, 'stats_level_posthoc.csv'), index=False)

print('Omnibus RM-ANOVA (level):')
p_col = 'p-GG-corr' if 'p-GG-corr' in aov_level.columns else 'p-unc'
display(aov_level[['Source', 'ddof1', 'ddof2', 'F', p_col, 'ng2']])
print('\nPost-hoc pairwise (FDR-BH):')
display(ph_level[['A', 'B', 'T', 'p-unc', 'hedges']])

# Box + strip plot, outline-only, one color per level (not L/R)
level_palette = dict(zip(level_order, sns.color_palette('Set2', len(level_order))))
fig, ax = plt.subplots(figsize=(5, 5))
sns.stripplot(data=df_level, x='level', y='PR', hue='level', order=level_order,
              hue_order=level_order, palette=level_palette, legend=False,
              linewidth=0.5, size=4, jitter=0.15, ax=ax, zorder=2)
sns.boxplot(data=df_level, x='level', y='PR', hue='level', order=level_order,
            hue_order=level_order, palette=level_palette, legend=False,
            linewidth=1, fliersize=0, ax=ax, zorder=1)
_outline_only(ax)

sig_pairs = ph_level[ph_level['p-unc'] < 0.05]
level_xi  = {l: i for i, l in enumerate(level_order)}
y_max = df_level['PR'].max()
for bx, (_, row) in enumerate(sig_pairs.iterrows()):
    x1, x2 = level_xi[row['A']], level_xi[row['B']]
    y = y_max + 0.5 + bx * 0.6
    ax.plot([x1, x1, x2, x2], [y, y + 0.2, y + 0.2, y], color='k', linewidth=1)
    p_label = '***' if row['p-unc'] < 0.001 else ('**' if row['p-unc'] < 0.01 else '*')
    ax.text((x1 + x2) / 2, y + 0.25, p_label, ha='center', fontsize=11)

ax.set_xticks(range(len(level_order)))
ax.set_xticklabels(level_order, fontsize=10)
ax.set_xlabel('')
ax.set_ylabel('Participation ratio (PR)')
ax.set_title('PR by hierarchical level')
sns.despine(ax=ax)
fig.tight_layout()
plt.savefig(os.path.join(out_dir, 'stats_level_boxplot.png'), dpi=600)
plt.show()


### 8d-ii. Voxel-Count Confound Check

PR is bounded by `min(n_stimuli, n_voxels)`. Before treating any Primary-vs-Association PR gap
(8d) as a real representational-hierarchy effect, check whether it's just a voxel-count ceiling
artifact.

In [ ]:
# Load per-subject, per-ROI voxel counts (saved by dimensionality_roi.py)
df_nvox = load_group_metric_df(out_dir, 'nvoxels')
print(f'Subjects loaded: {len(df_nvox)}')
df_nvox[roi_list].mean().round(1)


In [ ]:
# Scatter: mean n_voxels vs mean PR per ROI, colored by hierarchical level
roi_to_level = {roi: level for level, rois in level_map.items() for roi in rois}
mean_nvox = df_nvox[roi_list].mean()
mean_pr_roi = df_pr[roi_list].mean()

scatter_df = pd.DataFrame({
    'ROI': roi_list,
    'n_voxels': mean_nvox[roi_list].values,
    'PR': mean_pr_roi[roi_list].values,
    'level': [roi_to_level.get(r, 'NA') for r in roi_list],
})

fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=scatter_df, x='n_voxels', y='PR', hue='level',
                 hue_order=level_order, s=80, ax=ax)
for _, row in scatter_df.iterrows():
    ax.annotate(row['ROI'], (row['n_voxels'], row['PR']), fontsize=6,
                xytext=(3, 3), textcoords='offset points')
sns.despine(ax=ax)
fig.tight_layout()
plt.savefig(os.path.join(out_dir, 'voxel_count_vs_PR_scatter.png'), dpi=300)
plt.show()

corr = pg.corr(scatter_df['n_voxels'], scatter_df['PR'])
display(corr[['r', 'p-val', 'CI95%']])


In [ ]:
# ANCOVA: does the 'level' effect on PR survive after controlling for n_voxels?
ancova_df = (df_pr.melt(id_vars='subject', var_name='ROI', value_name='PR')
    .query('ROI in @roi_list')
    .merge(df_nvox.melt(id_vars='subject', var_name='ROI', value_name='n_voxels')
           .query('ROI in @roi_list'), on=['subject', 'ROI'])
    .assign(level=lambda d: d['ROI'].map(roi_to_level))
    .dropna())

ancova_res = pg.ancova(data=ancova_df, dv='PR', covar='n_voxels', between='level')
display(ancova_res)
print('If `level` remains significant here (compare to 8d), the Primary<->Association PR gap')
print('is unlikely to be purely a voxel-count ceiling artifact.')


## 9. Group Comparisons: Mandarin vs non-Mandarin

Test whether PR differs between language groups:
- **9a** Load group labels from `participants.tsv`
- **9b** Per-ROI independent t-tests (FDR-BH corrected)
- **9c** Mixed ANOVA: group (between-subjects) x ROI (within-subjects) interaction
- **9d** Grouped bar chart comparing the two groups per ROI
- **9e** Group comparison at hierarchical levels, incl. within-group Primary-vs-Association

In [ ]:
# 9a -- Group labels already in participants_df (Section 1); merge into df_pr
group_map = participants_df.set_index('subject')['group'].to_dict()

df_pr['group'] = df_pr['subject'].map(group_map)
df_pr_grp = df_pr.dropna(subset=['group']).copy()
print(df_pr_grp['group'].value_counts())


In [ ]:
# 9b -- Per-ROI independent t-tests (Mandarin vs non-Mandarin), FDR-BH corrected
rois_g = [r for r in roi_order if r in df_pr_grp.columns]
grp_results = []
for roi in rois_g:
    mand = df_pr_grp.loc[df_pr_grp['group'] == 'Mandarin', roi].dropna()
    nman = df_pr_grp.loc[df_pr_grp['group'] == 'non-Mandarin', roi].dropna()
    res = pg.ttest(mand, nman, paired=False)
    res.insert(0, 'ROI', roi)
    res['mean_Mandarin']    = mand.mean()
    res['mean_nonMandarin'] = nman.mean()
    grp_results.append(res)

df_grp = pd.concat(grp_results, ignore_index=True)
_, df_grp['p-corr'] = pg.multicomp(df_grp['p-val'].values, method='fdr_bh')
df_grp.to_csv(os.path.join(out_dir, 'stats_group_ttest.csv'), index=False)
display(df_grp[['ROI', 'mean_Mandarin', 'mean_nonMandarin', 'T', 'dof', 'p-val', 'p-corr', 'cohen-d']])


In [ ]:
# 9c -- Mixed ANOVA: group (between-subjects) x ROI (within-subjects)
df_long_grp = (df_pr_grp
    .melt(id_vars=['subject', 'group'], var_name='ROI', value_name='PR')
    .query('ROI in @roi_list')
    .dropna())

maov = pg.mixed_anova(data=df_long_grp, dv='PR', within='ROI',
                       between='group', subject='subject')
display(maov[['Source', 'F', 'DF1', 'DF2', 'p-unc', 'np2']])


In [ ]:
# 9d -- Grouped box+strip plot: Mandarin vs non-Mandarin PR per ROI (outline-only)
groups = ['Mandarin', 'non-Mandarin']
group_palette = dict(zip(groups, sns.color_palette('Set1', 2)))
rois_g = [r for r in roi_order if r in df_pr_grp.columns]

fig, ax = plt.subplots(figsize=(12, 5), dpi=200)
sns.stripplot(data=df_long_grp, x='ROI', y='PR', hue='group', order=rois_g,
              hue_order=groups, palette=group_palette, dodge=True, linewidth=0.5, size=3,
              legend=None, ax=ax, zorder=2)
sns.boxplot(data=df_long_grp, x='ROI', y='PR', hue='group', order=rois_g,
            hue_order=groups, palette=group_palette, dodge=True, linewidth=1, fliersize=0,
            ax=ax, zorder=1)
_outline_only(ax)

ax.legend(handles=[Patch(facecolor='none', edgecolor=group_palette[g], label=g) for g in groups],
          loc='upper left', frameon=False)
ax.set_xticklabels(rois_g, rotation=45, ha='right', fontsize=8)
ax.set_xlabel('')
ax.set_ylabel('Participation ratio (PR)')
ax.set_title('Representational dimensionality: Mandarin vs non-Mandarin')
sns.despine(ax=ax)
fig.tight_layout()
plt.savefig(os.path.join(out_dir, 'group_comparison_PR.png'), dpi=300)
plt.show()


In [ ]:
# 9e -- Group comparison at hierarchical levels: Mandarin vs non-Mandarin
df_level_grp = (pd.DataFrame(
    {level: df_pr_grp[[r for r in rois if r in df_pr_grp.columns]].mean(axis=1)
     for level, rois in level_map.items()}
).assign(subject=df_pr_grp['subject'].values,
          group=df_pr_grp['group'].values)
 .melt(id_vars=['subject', 'group'], var_name='level', value_name='PR')
 .dropna())

# Independent t-test per level, FDR-BH corrected
level_ttest = []
for level in level_order:
    sub  = df_level_grp[df_level_grp['level'] == level]
    mand = sub.loc[sub['group'] == 'Mandarin', 'PR']
    nman = sub.loc[sub['group'] == 'non-Mandarin', 'PR']
    res  = pg.ttest(mand, nman, paired=False)
    res.insert(0, 'level', level)
    res['mean_Mandarin']    = mand.mean()
    res['mean_nonMandarin'] = nman.mean()
    level_ttest.append(res)

df_level_ttest = pd.concat(level_ttest, ignore_index=True)
_, df_level_ttest['p-corr'] = pg.multicomp(df_level_ttest['p-val'].values, method='fdr_bh')
df_level_ttest.to_csv(os.path.join(out_dir, 'stats_group_level_ttest.csv'), index=False)
display(df_level_ttest[['level', 'mean_Mandarin', 'mean_nonMandarin', 'T', 'dof', 'p-val', 'p-corr', 'cohen-d']])


### 9e-ii. Primary vs. Association Within Each Group

Tests whether the Primary-vs-Association hierarchy itself flips direction by group, rather than
just pooling both groups together in 8d.

In [ ]:
# Formal interaction test: does level (Primary vs Association) depend on group?
maov_level_grp = pg.mixed_anova(data=df_level_grp, dv='PR', within='level',
                                 between='group', subject='subject')
display(maov_level_grp[['Source', 'F', 'DF1', 'DF2', 'p-unc', 'np2']])


In [ ]:
# Primary vs Association, paired t-test computed separately within each group
level_within_group_rows = []
for grp in groups:
    grp_df = df_level_grp[df_level_grp['group'] == grp]
    piv = grp_df.pivot(index='subject', columns='level', values='PR').dropna(subset=level_order)
    res = pg.ttest(piv['Primary'], piv['Association'], paired=True)
    res.insert(0, 'group', grp)
    res['mean_Primary']     = piv['Primary'].mean()
    res['mean_Association'] = piv['Association'].mean()
    level_within_group_rows.append(res)

df_level_within_group = pd.concat(level_within_group_rows, ignore_index=True)
_, df_level_within_group['p-corr'] = pg.multicomp(df_level_within_group['p-val'].values, method='fdr_bh')
df_level_within_group.to_csv(os.path.join(out_dir, 'stats_level_within_group.csv'), index=False)
display(df_level_within_group[['group', 'mean_Primary', 'mean_Association', 'T', 'dof',
                                'p-val', 'p-corr', 'cohen-d']])


In [ ]:
# 9f -- Grouped box+strip plot: Mandarin vs non-Mandarin per hierarchical level (outline-only)
fig, ax = plt.subplots(figsize=(7, 5), dpi=200)
sns.stripplot(data=df_level_grp, x='level', y='PR', hue='group', order=level_order,
              hue_order=groups, palette=group_palette, dodge=True, linewidth=0.5, size=4,
              legend=None, ax=ax, zorder=2)
sns.boxplot(data=df_level_grp, x='level', y='PR', hue='group', order=level_order,
            hue_order=groups, palette=group_palette, dodge=True, linewidth=1, fliersize=0,
            ax=ax, zorder=1)
_outline_only(ax)

# Significance brackets: between-group comparison at each level (black)
sig_rows = df_level_ttest[df_level_ttest['p-corr'] < 0.05]
y_max = df_level_grp['PR'].max()
level_xi = {l: i for i, l in enumerate(level_order)}
dodge_offset = 0.2
bx = 0
for _, row in sig_rows.iterrows():
    xi = level_xi[row['level']]
    x1, x2 = xi - dodge_offset, xi + dodge_offset
    y = y_max + 0.2 + bx * 0.6
    ax.plot([x1, x1, x2, x2], [y, y + 0.2, y + 0.2, y], color='k', linewidth=1)
    p_label = '***' if row['p-corr'] < 0.001 else ('**' if row['p-corr'] < 0.01 else '*')
    ax.text((x1 + x2) / 2, y + 0.25, p_label, ha='center', fontsize=11)
    bx += 1

# Significance brackets: within-group Primary-vs-Association level difference (color-coded)
hue_offset = {groups[0]: -dodge_offset, groups[1]: dodge_offset}
sig_level_rows = df_level_within_group[df_level_within_group['p-corr'] < 0.05]
for _, row in sig_level_rows.iterrows():
    offset = hue_offset[row['group']]
    x1, x2 = level_xi['Primary'] + offset, level_xi['Association'] + offset
    y = y_max + 0.2 + bx * 0.3
    color = group_palette[row['group']]
    ax.plot([x1, x1, x2, x2], [y, y + 0.2, y + 0.2, y], color=color, linewidth=1)
    p_label = '***' if row['p-corr'] < 0.001 else ('**' if row['p-corr'] < 0.01 else '*')
    ax.text((x1 + x2) / 2, y + 0.25, p_label, ha='center', fontsize=11, color=color)
    bx += 1

ax.legend(handles=[Patch(facecolor='none', edgecolor=group_palette[g], label=g) for g in groups],
          loc='upper right', frameon=False,
         bbox_to_anchor=(1.3, 0.9))
ax.set_xticks(range(len(level_order)))
ax.set_xticklabels(level_order, fontsize=10)
ax.set_xlabel('')
ax.set_ylabel('Participation ratio (PR)')
ax.set_title('PR by hierarchical level: Mandarin vs non-Mandarin')
sns.despine(ax=ax)
fig.tight_layout()
plt.savefig(os.path.join(out_dir, 'group_level_comparison.png'), dpi=300)
plt.show()


In [ ]:
print(out_dir)
